In [48]:
from langgraph.graph import StateGraph ,START,END
from langchain_groq import ChatGroq
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel , Field

In [49]:
load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [50]:
class SentimentSchema(BaseModel):

  sentiment: Literal["positive","negative"] = Field(description='Sentiment of the review')

In [51]:
class DiagnosisSchema(BaseModel):

  issue_type: Literal["UX","Performance","Bug","Support","Other"] = Field(description='The category of issue mentioned in the review')
  tone:Literal["angry","frustrated","disappointed","calm"] = Field(description="The emotional tone expressed by the user.")
  urgency :Literal["low","medium","high"] = Field(description="How urgent or critical the issue appears to be.")

In [52]:
structured_model = model.with_structured_output(SentimentSchema)

structured_model2 = model.with_structured_output(DiagnosisSchema)

In [53]:
prompt = "What is the sentiment of the following review - The software too bad"

structured_model.invoke(prompt).sentiment

'negative'

In [54]:
class ReviewState(TypedDict):

  review: str
  sentiment: Literal['positive','negative']
  diagnosis:dict
  response:str

In [55]:
def find_sentiment(state:ReviewState):

  prompt = f"For the following review find out the sentiment \n {state['review']}"
  sentiment = structured_model.invoke(prompt).sentiment

  return {'sentiment':sentiment}

def check_sentiment(state:ReviewState) -> Literal['positive_response',"run_diagnosis"]:

  if state['sentiment'] == 'positive':
    return 'positive_response'
  else:
    return 'run_diagnosis'

In [60]:
def positive_response(state:ReviewState):

  prompt = f"""Write a warm thank-you message in response to this review:
  \n\n"{state['review']}\"\n
  Also ,kindly ask the user to leave feedback on our website."""

  response = model.invoke(prompt).content

  return {'response':response}


def run_diagnosis(state:ReviewState):

  prompt = f"""Diagnose this negative review :\n\n{state['review']}\n"
  "Return issue_type, tone and urgency.
"""

  response = structured_model2.invoke(prompt)

  return {'diagnosis':response.model_dump()} # model_dump create a dictionary object from the json - response which is in json format

def negative_response(state:ReviewState):

  diagnosis = state['diagnosis']

  prompt = f"""You are a support assistance.
  The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
  Write an empathatic ,helpful resolution message.
  
"""
  response = model.invoke(prompt).content

  return {'response':response}


In [61]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('negative_response',negative_response)


graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)
graph.add_edge('positive_response',END)
graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('negative_response',END)

workflow = graph.compile()


In [63]:
initial_state = {'review':'The product is absolutely amazing! The quality is excellent, and it works exactly as expected. The design is attractive, the features are easy to use, and the overall performance is impressive. It offers great value for money, and I am very satisfied with my purchase. I would definitely recommend this product to anyone looking for something reliable and high-quality!'}

workflow.invoke(initial_state)

{'review': 'The product is absolutely amazing! The quality is excellent, and it works exactly as expected. The design is attractive, the features are easy to use, and the overall performance is impressive. It offers great value for money, and I am very satisfied with my purchase. I would definitely recommend this product to anyone looking for something reliable and high-quality!',
 'sentiment': 'positive',
 'response': 'Dear [Customer’s Name],\n\nThank you so much for taking the time to share such a wonderful review! We’re thrilled to hear that you find the product’s quality, design, and performance outstanding, and that it delivers the value and reliability you were looking for. Your kind words truly make our day and motivate us to keep delivering high‑quality, user‑friendly products.\n\nIf you have a moment, we’d love for you to share your experience on our website as well. Your feedback helps other shoppers make confident decisions and lets us continue improving our offerings.\n\nTh